# PPR10K Download & Prep (for ECCV rebuttal)

**目的**: 把 PPR10K-360p 下载到 `MyDrive/datasets/PPR10K/`,并整理成 `train/{input,gt}` + `val/{input,gt}` 结构,与 `data/paired_folder.py` 兼容。

**用法**:
1. 跑 Cell 1 挂载 Drive + 装 gdown
2. 打开 https://github.com/csjliang/PPR10K → README 里找 "Downloads" → 复制 360p 版本的 Google Drive 链接
3. 把链接粘到 Cell 2 的变量里
4. 跑 Cell 3-5

**预期大小**: 360p 版本 ~7-10GB,在 Colab → Drive 高带宽下应该 5-15 分钟搞定。

In [ ]:
# === Cell 1: mount + install gdown ===
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gdown

import os
DEST = '/content/drive/MyDrive/datasets/PPR10K'
os.makedirs(DEST, exist_ok=True)
%cd /content

In [ ]:
# === Cell 2: 配置下载 URL ===
# 去 https://github.com/csjliang/PPR10K 找 "360p paired data" 的 Google Drive 链接
# 形如 https://drive.google.com/file/d/<FILE_ID>/view 或 https://drive.google.com/drive/folders/<FOLDER_ID>
# 把 ID 填到下面

# 选你想训练的 retoucher (a / b / c — PPR10K 有 3 个专家)
EXPERT = 'a'  # 大多数论文用 expert a

# 从 PPR10K README 抄过来的 file/folder IDs (训练 + 测试各一份)
DOWNLOADS = {
    'train_input_360p.zip':   '<FILE_ID_HERE>',  # 训练输入
    f'train_target_{EXPERT}_360p.zip': '<FILE_ID_HERE>',  # 训练 GT
    'test_input_360p.zip':    '<FILE_ID_HERE>',  # 测试输入
    f'test_target_{EXPERT}_360p.zip':  '<FILE_ID_HERE>',  # 测试 GT
}

print(f"Will download {len(DOWNLOADS)} files to {DEST}/raw/")
print("Expert:", EXPERT)
for k, v in DOWNLOADS.items():
    status = '✅' if not v.startswith('<') else '❌ TODO'
    print(f"  {status} {k}: {v}")

In [ ]:
# === Cell 3: download via gdown (直接下到 Drive) ===
import gdown, os

RAW_DIR = f'{DEST}/raw'
os.makedirs(RAW_DIR, exist_ok=True)

for filename, file_id in DOWNLOADS.items():
    if file_id.startswith('<'):
        print(f"⏭️  Skip {filename} (no ID)")
        continue
    out_path = f'{RAW_DIR}/{filename}'
    if os.path.exists(out_path):
        print(f"⏭️  {filename} already exists ({os.path.getsize(out_path)/1e9:.2f} GB)")
        continue
    print(f"⬇️  Downloading {filename} ...")
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, out_path, quiet=False)
    print(f"   ✅ {os.path.getsize(out_path)/1e9:.2f} GB")

!ls -lh {RAW_DIR}

In [ ]:
# === Cell 4: extract zips ===
import zipfile, os, glob

EXTRACT_DIR = f'{DEST}/extracted'
os.makedirs(EXTRACT_DIR, exist_ok=True)

for zf in sorted(glob.glob(f'{RAW_DIR}/*.zip')):
    name = os.path.basename(zf).replace('.zip', '')
    out_dir = f'{EXTRACT_DIR}/{name}'
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) > 0:
        print(f"⏭️  {name} already extracted ({len(os.listdir(out_dir))} files)")
        continue
    print(f"📦 Extracting {name} ...")
    with zipfile.ZipFile(zf) as z:
        z.extractall(out_dir)
    n = sum(len(files) for _, _, files in os.walk(out_dir))
    print(f"   ✅ {n} files extracted")

!ls {EXTRACT_DIR}/

In [ ]:
# === Cell 5: 整理成 train/{input,gt} + val/{input,gt} 结构 ===
# 这是 data/paired_folder.py 期望的格式
import os, glob, shutil

FINAL_DIR = f'{DEST}/expert_{EXPERT}'  # 最终训练用的路径
for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    os.makedirs(f'{FINAL_DIR}/{sub}', exist_ok=True)

# PPR10K 解压后内部结构需要确认 — 解压完看一下:
print("=== extracted 内部结构 (debug用) ===")
for d in sorted(glob.glob(f'{EXTRACT_DIR}/*')):
    files = []
    for root, _, fs in os.walk(d):
        for f in fs[:3]:
            files.append(os.path.join(root, f).replace(EXTRACT_DIR, '...'))
    print(f"📂 {os.path.basename(d)}")
    for f in files[:5]:
        print(f"   {f}")

# 根据上面的输出,改下面这 4 行的 SOURCE 路径(看 PPR10K zip 解压后实际放在哪个子目录)
MAPPINGS = [
    (f'{EXTRACT_DIR}/train_input_360p',         f'{FINAL_DIR}/train/input'),
    (f'{EXTRACT_DIR}/train_target_{EXPERT}_360p', f'{FINAL_DIR}/train/gt'),
    (f'{EXTRACT_DIR}/test_input_360p',          f'{FINAL_DIR}/val/input'),
    (f'{EXTRACT_DIR}/test_target_{EXPERT}_360p',  f'{FINAL_DIR}/val/gt'),
]

for src, dst in MAPPINGS:
    if not os.path.exists(src):
        print(f"⚠️  {src} 不存在 — 检查一下解压后 PPR10K 的实际目录名")
        continue
    # 找 src 内部的图片(可能直接在根,也可能多套一层)
    imgs = glob.glob(f'{src}/**/*.tif', recursive=True) + \
           glob.glob(f'{src}/**/*.png', recursive=True) + \
           glob.glob(f'{src}/**/*.jpg', recursive=True)
    print(f"📁 {src} → {dst}: {len(imgs)} files")
    for img in imgs:
        try:
            os.symlink(img, f'{dst}/{os.path.basename(img)}')
        except FileExistsError:
            pass

print("\n=== 最终结构 ===")
for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    n = len(os.listdir(f'{FINAL_DIR}/{sub}'))
    print(f"  {FINAL_DIR}/{sub}: {n} files")

In [ ]:
# === Cell 6: 验证 — 用我们的 Dataset 加载几张看看 ===
import sys
sys.path.insert(0, '/content/drive/MyDrive/LoR-LUT')
from data.paired_folder import PairedFolderDataset

ds = PairedFolderDataset(
    root=FINAL_DIR,
    split='train',
    in_dir='input',
    gt_dir='gt',
    patch=0,
    augment=False
)
print(f"✅ {len(ds)} paired samples in train")
for i in range(3):
    s = ds[i]
    diff = (s['img_in'] - s['img_gt']).abs().mean().item()
    print(f"  [{i}] {s['name']:30s} input-gt MAE={diff:.4f} (should be > 0)")

## 完成后

训练 PPR10K 时:
```python
!python train.py \
    --cfg config/default.yaml \
    --data.root /content/drive/MyDrive/datasets/PPR10K/expert_a \
    --work_dir /content/drive/MyDrive/LoR-LUT/runs/ppr10k_a_K0_R8
```

## 注意

- `Cell 5` 用了 `os.symlink` 而不是 `shutil.copy` — 节省 2x 空间,但要确保 Drive 同步起来 (一般可以)
- 如果训练时报错说找不到符号链接的目标,改成 `shutil.copy`(把 Cell 5 里 `os.symlink` 换掉)
- 如果 PPR10K 解压后的目录名跟我猜的不一样 (`train_input_360p` 之类),Cell 5 会提示,你改 `MAPPINGS` 里的路径